In [12]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings 
from pathlib import Path
import frontmatter

In [7]:
def load_markdown_folder(folder_path):
    documents = []

    for file_path in Path(folder_path).rglob("*.md"):   

        post = frontmatter.load(file_path)

        metadata = dict(post.metadata)

        content = post.content

        documents.append({
            "content": content,
            "metadata": {
                **metadata,
                "source": str(file_path),
                "file_name": file_path.name,
                "file_type": "markdown"
            }
        })

    return documents


documents = load_markdown_folder("./knowledge-base")

In [15]:
ddocs = []
for doc in documents:
    ddocs.append(
        Document(
            page_content=doc["content"],
            metadata=doc["metadata"]
        )
    )


In [ ]:
split = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

In [ ]:
split_docs = split.split_documents(ddocs)

VECTOR DB

In [19]:
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

In [21]:
embeddings = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-m3"
)

In [22]:
client = QdrantClient(path="qdrant.db")

In [23]:
collection_name = "knowlegedb"

In [24]:
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)


True

In [25]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
)

In [26]:
vector_store.add_documents(split_docs)
# results = vector_store.similarity_search("pyhton", k=3)

['deb155e9c82944cca9d011f84a83dd82',
 '64178c77596b4bcabc8d77c8d8fe0e3a',
 '71c03972ff42430ea08023ce11bb7245',
 '053d858661774401825df46e4e306fba',
 'cb6ce924fd68482c83cfc009630c9a0a',
 '4975ab46e5f64244b30fbf1e37e028c3',
 '3ae66c2131874dbfa1c31283ec53362d',
 'dfbccf304b8c478aae3febfbf6ae0e9c',
 '86bb01489d0147598db809e93b6bdf73',
 'c8548a9ec3bf454db3ceb7cacc8689dd',
 '507fa4ea22a1466b93940d7629740465',
 'aab6d49a9fd746a5925087422d1c4455',
 '97ea1d80ff414afca57b4b7e86aaabc3',
 '9ff8f99cd81e4e7aa7b893144b62f57d',
 '71a479009f424007a082dcc40cafd82a',
 'ffc59a360e5744a2b4133de3f76c3631',
 'e9c2c71257c34c8c8460fdc71358885c',
 '432b2fdb78a64e13847e515d30a52709',
 '507ca848c64e464b90c46a462bfb3126',
 '772794237b5e4ceab4d55568d8c90794',
 'ab3a644534e3482bbe6ee3dc7e9b1bd6',
 '3da329b8930944ffbd24aed5f82fbc5e',
 '628bb770faa442cb94d68310828d0499',
 'd44377a660ef4fc486865d5424b8e9f1',
 '697259e8f21c445d916741d96d486961',
 '609f4f12e9d54f95b19ddb6c96115997',
 'fffea179a1ad498da8baacbc9a86df51',
 

In [15]:
vector_store

Query

In [27]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, MultiQueryRetriever
from langchain_groq import ChatGroq

C:\Users\shega\AppData\Local\Temp\ipykernel_29312\1937584044.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [36]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)


In [29]:
k=5

Hybrid Search + Multi Query Retriver

In [ ]:
bm25_retriever = BM25Retriever.from_documents(split_docs,k=k) 
retriever = vector_store.as_retriever(search_kwargs={"k": k}) 

ensemble_retriever = EnsembleRetriever(retrievers=[retriever, bm25_retriever], weights=[0.5, 0.5])

multi_query_retriever = MultiQueryRetriever.from_llm(
        retriever=ensemble_retriever,
        llm=llm
    )

In [32]:
from langchain_core.messages import HumanMessage, SystemMessage,AIMessage

In [33]:
query = "My TrailPlus membership was active when I ordered. What is my return window?"

In [37]:
def HYDE(query: str) -> str:
    messages = [
        SystemMessage(content="""You are generating a hypothetical document to improve retrieval for a search system.
Given the following question, write a short passage that would plausibly contain the answer.
Write it as if it were an excerpt from a real document (e.g. a research paper, article, or technical report) — not as a direct answer to the question, and not addressed to the reader.
Do not hedge, do not say "I don't know," and do not include disclaimers.
Write confidently, using domain-appropriate terminology, even if some details are invented.
Keep it to 3-5 sentences."""),
        HumanMessage(content=query)
    ]
    response = llm.invoke(messages)
    return response.content
    

In [38]:
enhanced_query=HYDE(query)

In [39]:
enhanced_query

'TrailPlus members enjoy an extended return policy that differs from the standard consumer window. When a purchase is made while the membership is active, the return period is calculated from the date of delivery rather than the order date. Specifically, the eligible window for a full refund or exchange is sixty (60) calendar days after the customer receives the item. This extended timeframe applies to all eligible merchandise purchased under the TrailPlus program, provided the product is returned in its original condition with proof of purchase.'

In [42]:

unique_docs = multi_query_retriever.invoke(enhanced_query)

In [43]:
seen = set()
unique_clean_docs = []
for doc in unique_docs:
    if doc.page_content not in seen:
        seen.add(doc.page_content)
        unique_clean_docs.append(doc.page_content)

Reranking

In [44]:
import cohere

In [45]:
load_dotenv()
reranking_model = cohere.ClientV2()
response = reranking_model.rerank(
        model="rerank-v3.5",
        query=query,
        documents=unique_clean_docs,
        top_n=7
    )

In [46]:
reranked_docs = []

for result in response.results:
        doc = unique_clean_docs[result.index]
        reranked_docs.append(doc)

In [47]:
reranked_docs

['# TrailPlus Membership Benefits\n\n## Return window\n\nA customer whose TrailPlus membership was active when an order was placed receives a **45-calendar-day return window from delivery** for eligible items.\n\nJoining TrailPlus after placing an order does not extend that order’s return window.\n\nFinal-sale restrictions, item-condition requirements, and warranty rules still apply.\n\n## Shipping benefit\n\nTrailPlus members receive free standard shipping on eligible United States orders without a minimum purchase amount.\n\nThe benefit does not cover expedited shipping, import duties, Canadian return postage, or other international charges.\n\n## Membership verification\n\nWhen membership status is not available to the agent, it should explain the standard policy and ask the customer to confirm whether TrailPlus was active on the order date. It must not assume membership based only on the customer requesting the benefit.',
 '# TrailPlus Membership Benefits\n\n## Return window\n\nA c

In [48]:
response.results

[V2RerankResponseResultsItem(index=3, relevance_score=0.9466806),
 V2RerankResponseResultsItem(index=2, relevance_score=0.94632465),
 V2RerankResponseResultsItem(index=1, relevance_score=0.91490877),
 V2RerankResponseResultsItem(index=0, relevance_score=0.9132521),
 V2RerankResponseResultsItem(index=4, relevance_score=0.6350646),
 V2RerankResponseResultsItem(index=6, relevance_score=0.4457955),
 V2RerankResponseResultsItem(index=10, relevance_score=0.40161797)]

In [49]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}
""")

# Use reranked_docs directly — they are already strings (page_content)
context = "\n\n".join(reranked_docs)

chain = prompt | llm | StrOutputParser()


In [50]:
answer = chain.invoke({"context": context, "question": query})
print(answer)

Your return window is **45 calendar days from the delivery date** for eligible items.
